# M2 N/V 축별 비음수 게이트 — Dunnhumby seed 42

기존 `ID|N|V` 공동학습 구조는 그대로 둡니다. 거래활동 수준 `q_N` 은 N 블록에만, 거래당 가치 수준 `q_V` 는 V 블록에만 비음수·평균 1 게이트로 반영합니다. M1과 M2만 같은 seed·학습예산·선택규칙으로 학습하며 validation만 확인합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess

REVIEWED_SHA = '4accfe76ace0c8bbd71adf936b66e8003160ec31'
%cd /content
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
%cd /content/clv-m2-lightgcn-runner
print('검토 코드 고정 완료:', actual_sha)

In [ ]:
import json, torch
from lightgcn_clv_axis_specific_gate import (
    configure_axis_specific_gate_dunnhumby_run,
    preflight_summary,
    run_experiment,
)

cfg = configure_axis_specific_gate_dunnhumby_run()
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
result_df = run_experiment(cfg)

In [ ]:
from IPython.display import display

columns = [
    'selection_rule', 'model_id', 'role', 'gate_shape', 'selected_epoch',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'revenue@10', 'revenue@20', 'revenue@50', 'arp@10',
    'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
    'eff_catalog@10', 'top10_share@10', 'top100_share@10',
    'value_alignment', 'gamma_n', 'gamma_v',
    'gate_n_mean', 'gate_n_std', 'gate_n_negative_share',
    'gate_v_mean', 'gate_v_std', 'gate_v_negative_share',
]
available = [column for column in columns if column in result_df.columns]
display(result_df[available].sort_values(['selection_rule', 'model_id']))
print('최종 판정:')
print(json.dumps(result_df.attrs['screening_decision'], ensure_ascii=False, indent=2))
print('동일 정확도 하한 기준:', result_df.attrs['guard_reference']['thresholds_99pct'])
print('결과 파일:', result_df.attrs['result_paths'])